In [0]:

from datetime import datetime

modo = "historico" # "automatico"


if modo == "automatico":
  periodo = datetime.now().strftime("%m-%Y")
else :
  periodo = None

  

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.financiero.fact_transacciones (
     ID_MOVIMIENTO            STRING,
     ID_CLIENTE               STRING,
     CODIGO_PRODUCTO          STRING,
     NUMERO_CUENTA            STRING,
     FECHA_MOVIMIENTO         DATE,
     HORA_MOVIMIENTO          STRING,
     MONTO_COP                DOUBLE,
     MONTO_USD                DOUBLE,
     TIPO_MOVIMIENTO          STRING,
     ESTADO_MOVIMIENTO        STRING,
     CODIGO_CANAL             STRING,
     CODIGO_CIUDAD            STRING,
     FLAG_HORARIO             STRING,
     PROMEDIO_MOVIL_30D       DOUBLE,
     FLAG_ANOMALIA            INT,
     PERIODO                  STRING,
     _FECHA_CARGA             DATE,
     _FUENTE                  STRING
)
USING DELTA
PARTITIONED BY (periodo)
LOCATION 'abfss://gold@stdataknowdeveastus001.dfs.core.windows.net/financiero/fact_transacciones';

In [0]:
if modo == "automatico":
    spark.sql(f"""
        CREATE OR REPLACE TEMPORARY VIEW tvw_fact_transacciones AS
        SELECT
             m.id_mov                                                        AS ID_MOVIMIENTO
            ,m.id_cli                                                        AS ID_CLIENTE
            ,m.cod_prod                                                      AS CODIGO_PRODUCTO
            ,m.num_cuenta                                                    AS NUMERO_CUENTA
            ,m.cod_canal                                                     AS CODIGO_CANAL
            ,m.cod_ciudad                                                    AS CODIGO_CIUDAD
            ,m.fec_mov                                                       AS FECHA_MOVIMIENTO
            ,m.hra_mov                                                       AS HORA_MOVIMIENTO
            ,m.vr_mov                                                        AS MONTO_COP
            ,ROUND(m.vr_mov / 4200, 2)                                       AS MONTO_USD
            ,m.tip_mov                                                       AS TIPO_MOVIMIENTO
            ,m.cod_estado_mov                                                AS ESTADO_MOVIMIENTO
            ,CASE
                WHEN DAYOFWEEK(m.fec_mov) BETWEEN 2 AND 6
                 AND CAST(SPLIT(m.hra_mov, ':')[0] AS INT) BETWEEN 8 AND 17
                THEN 'HABIL'
                ELSE 'NO HABIL'
             END                                                             AS FLAG_HORARIO
            ,ROUND(AVG(m.vr_mov) OVER (
                PARTITION BY m.id_cli
                ORDER BY CAST(m.fec_mov AS TIMESTAMP)
                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND CURRENT ROW
            ), 2)                                                            AS PROMEDIO_MOVIL_30D
            ,CASE
                WHEN m.vr_mov > 3 * AVG(m.vr_mov) OVER (
                    PARTITION BY m.id_cli
                    ORDER BY CAST(m.fec_mov AS TIMESTAMP)
                    RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND CURRENT ROW
                ) THEN 1
                ELSE 0
             END                                                             AS FLAG_ANOMALIA
            ,m.periodo                                                       AS PERIODO
            ,CURRENT_DATE                                                    AS _FECHA_CARGA
            ,'silver.cleaned.tb_mov_financieros'                             AS _FUENTE
        FROM silver.cleaned.tb_mov_financieros m
        INNER JOIN gold.financiero.dim_clientes c
            ON m.id_cli = c.ID_CLIENTE
        WHERE m.periodo = '{periodo}'
    """)

    spark.sql(f"""
        DELETE FROM gold.financiero.fact_transacciones
        WHERE PERIODO = '{periodo}'
    """)

    spark.sql("""
        INSERT INTO gold.financiero.fact_transacciones
        SELECT
             ID_MOVIMIENTO
            ,ID_CLIENTE
            ,CODIGO_PRODUCTO
            ,NUMERO_CUENTA
            ,FECHA_MOVIMIENTO
            ,HORA_MOVIMIENTO
            ,MONTO_COP
            ,MONTO_USD
            ,TIPO_MOVIMIENTO
            ,ESTADO_MOVIMIENTO
            ,CODIGO_CANAL
            ,CODIGO_CIUDAD
            ,FLAG_HORARIO
            ,PROMEDIO_MOVIL_30D
            ,FLAG_ANOMALIA
            ,PERIODO
            ,_FECHA_CARGA
            ,_FUENTE
        FROM tvw_fact_transacciones
    """)

In [0]:
%sql
SELECT DISTINCT PERIODO FROM silver.cleaned.tb_mov_financieros

In [0]:
periodos = spark.sql(f"""
    SELECT DISTINCT PERIODO FROM silver.cleaned.tb_mov_financieros
""")
periodos = periodos.select("PERIODO").collect()
# periodos = [periodo.PERIODO for periodo in periodos]
print(periodos)
    

In [0]:
if modo == "historico":
    periodos = spark.sql(f"""
      SELECT DISTINCT PERIODO FROM silver.cleaned.tb_mov_financieros
    """)
    periodos = periodos.select("PERIODO").collect()
    periodos = [periodo.PERIODO for periodo in periodos]
    for periodo in periodos:

        spark.sql(f"""
            CREATE OR REPLACE TEMPORARY VIEW tvw_fact_transacciones AS
            SELECT
                m.id_mov                                                        AS ID_MOVIMIENTO
                ,m.id_cli                                                        AS ID_CLIENTE
                ,m.cod_prod                                                      AS CODIGO_PRODUCTO
                ,m.num_cuenta                                                    AS NUMERO_CUENTA
                ,m.cod_canal                                                     AS CODIGO_CANAL
                ,m.cod_ciudad                                                    AS CODIGO_CIUDAD
                ,m.fec_mov                                                       AS FECHA_MOVIMIENTO
                ,m.hra_mov                                                       AS HORA_MOVIMIENTO
                ,m.vr_mov                                                        AS MONTO_COP
                ,ROUND(m.vr_mov / 4200, 2)                                       AS MONTO_USD
                ,m.tip_mov                                                       AS TIPO_MOVIMIENTO
                ,m.cod_estado_mov                                                AS ESTADO_MOVIMIENTO
                ,CASE
                    WHEN DAYOFWEEK(m.fec_mov) BETWEEN 2 AND 6
                    AND CAST(SPLIT(m.hra_mov, ':')[0] AS INT) BETWEEN 8 AND 17
                    THEN 'HABIL'
                    ELSE 'NO HABIL'
                END                                                             AS FLAG_HORARIO
                ,ROUND(AVG(m.vr_mov) OVER (
                    PARTITION BY m.id_cli
                    ORDER BY CAST(m.fec_mov AS TIMESTAMP)
                    RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND CURRENT ROW
                ), 2)                                                            AS PROMEDIO_MOVIL_30D
                ,CASE
                    WHEN m.vr_mov > 3 * AVG(m.vr_mov) OVER (
                        PARTITION BY m.id_cli
                        ORDER BY CAST(m.fec_mov AS TIMESTAMP)
                        RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND CURRENT ROW
                    ) THEN 1
                    ELSE 0
                END                                                             AS FLAG_ANOMALIA
                ,m.periodo                                                       AS PERIODO
                ,CURRENT_DATE                                                    AS _FECHA_CARGA
                ,'silver.cleaned.tb_mov_financieros'                             AS _FUENTE
            FROM silver.cleaned.tb_mov_financieros m
            INNER JOIN gold.financiero.dim_clientes c
                ON m.id_cli = c.ID_CLIENTE
            WHERE m.periodo = '{periodo}'
        """)

        spark.sql(f"""
            DELETE FROM gold.financiero.fact_transacciones
            WHERE PERIODO = '{periodo}'
        """)

        spark.sql("""
            INSERT INTO gold.financiero.fact_transacciones
            SELECT
                ID_MOVIMIENTO
                ,ID_CLIENTE
                ,CODIGO_PRODUCTO
                ,NUMERO_CUENTA
                ,FECHA_MOVIMIENTO
                ,HORA_MOVIMIENTO
                ,MONTO_COP
                ,MONTO_USD
                ,TIPO_MOVIMIENTO
                ,ESTADO_MOVIMIENTO
                ,CODIGO_CANAL
                ,CODIGO_CIUDAD
                ,FLAG_HORARIO
                ,PROMEDIO_MOVIL_30D
                ,FLAG_ANOMALIA
                ,PERIODO
                ,_FECHA_CARGA
                ,_FUENTE
            FROM tvw_fact_transacciones
        """)